# Vegetation Recovery Monitoring — Mount Arthur Coal Mine, NSW
### Google Earth Engine Time-Series Analysis | Python | Sentinel-2

This notebook monitors vegetation recovery on rehabilitated mine landforms
at Mount Arthur Coal Mine using Sentinel-2 NDVI time-series analysis
in Google Earth Engine. See README.md for full project documentation.

In [ ]:
# Cell 1 — Mount Google Drive to access AOI data and save outputs

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cell 2 — Update geemap to latest version for full feature support

%pip install -U geemap


In [ ]:
# Cell 3 — Import all required libraries for GEE, spatial analysis and visualisation

import ee
import geemap
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import os


In [ ]:
# Cell 4 — Authenticate and initialise Google Earth Engine with cloud project

ee.Authenticate()
ee.Initialize(project='portfolio2-499414')

In [ ]:
# Cell 5 — Load AOI polygons from Drive and assign zone labels

gdf = gpd.read_file('/content/drive/MyDrive/GIS Portfolio/Project1_MineRehab/Data/mount_arthur_aoi.geojson')
gdf['zone'] = ['rehab_mature', 'mining_clearance', 'rehab_active', 'reference']
print(gdf)


In [ ]:
# Cell 6 — Convert AOI to Earth Engine FeatureCollection and extract zone geometries

aoi_fc = geemap.geopandas_to_ee(gdf)

rehab_mature_geom = aoi_fc.filter(ee.Filter.eq('zone', 'rehab_mature')).geometry()
rehab_active_geom = aoi_fc.filter(ee.Filter.eq('zone', 'rehab_active')).geometry()
reference_geom = aoi_fc.filter(ee.Filter.eq('zone', 'reference')).geometry()
mining_clearance_geom = aoi_fc.filter(ee.Filter.eq('zone', 'mining_clearance')).geometry()

print('Rehab mature (m²):', rehab_mature_geom.area().getInfo())
print('Rehab active (m²):', rehab_active_geom.area().getInfo())
print('Reference (m²):', reference_geom.area().getInfo())
print('Mining clearance (m²):', mining_clearance_geom.area().getInfo())

In [ ]:
# Cell 7 — Define cloud masking and spectral index functions with property preservation

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
        qa.bitwiseAnd(cirrus_bit).eq(0)
    )
    masked = image.updateMask(mask).divide(10000)
    return masked.copyProperties(image, ['system:time_start'])

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    bsi = image.expression(
        '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))',
        {
            'SWIR1': image.select('B11'),
            'RED': image.select('B4'),
            'NIR': image.select('B8'),
            'BLUE': image.select('B2')
        }
    ).rename('BSI')
    result = image.addBands([ndvi, bsi])
    return result.copyProperties(image, ['system:time_start'])


In [ ]:
# Cell 8 — Filter Sentinel-2 collection to AOI, apply cloud mask and compute indices

start_date = '2017-01-01'
end_date = '2026-01-01'
cloud_thresh = 20

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi_fc.geometry())
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_thresh))
)

s2_masked = s2_collection.map(mask_s2_clouds)
s2_indexed = s2_masked.map(add_indices)

# Verify timestamps are present
info = s2_indexed.limit(3).getInfo()
for f in info['features']:
    print(f['properties'].get('system:time_start'))

print('Total images:', s2_indexed.size().getInfo())


In [ ]:
# Cell 9 — Build annual median composites for each year 2017-2025

years = list(range(2017, 2026))

def get_annual_composite(year):
    start = f'{year}-01-01'
    end = f'{year}-12-31'
    composite = (
        s2_indexed
        .filterDate(start, end)
        .median()
        .clip(aoi_fc.geometry())
        .set('year', year)
    )
    return composite

annual_composites = [get_annual_composite(y) for y in years]

# Verify bands present in each composite
for composite, year in zip(annual_composites, years):
    bands = composite.bandNames().getInfo()
    print(year, '-', bands)


In [ ]:
# Cell 10 — Define zonal statistics function and zones dictionary for all four AOI zones

def get_zonal_stats(image, geometry, zone_name, year):
    stats = image.select(['NDVI', 'BSI']).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=10,
        maxPixels=1e9
    ).getInfo()
    return {
        'year': year,
        'zone': zone_name,
        'NDVI': stats.get('NDVI'),
        'BSI': stats.get('BSI')
    }

zones = {
    'rehab_mature': rehab_mature_geom,
    'rehab_active': rehab_active_geom,
    'reference': reference_geom,
    'mining_clearance': mining_clearance_geom
}


In [ ]:
# Cell 11 — Extract mean NDVI and BSI per zone per year across all annual composites

results = []

for composite, year in zip(annual_composites, years):
    for zone_name, geom in zones.items():
        row = get_zonal_stats(composite, geom, zone_name, year)
        results.append(row)
        print(row)


In [ ]:
# Cell 12 — Convert zonal statistics results to pandas DataFrame for analysis

df = pd.DataFrame(results)
print(df)


In [ ]:
# Cell 13 — Plot NDVI time-series trend chart for all zones 2017-2025 and save to Drive

fig, ax = plt.subplots(figsize=(10, 6))

for zone_name in df['zone'].unique():
    subset = df[df['zone'] == zone_name]
    ax.plot(subset['year'], subset['NDVI'], marker='o', label=zone_name)

ax.set_xlabel('Year')
ax.set_ylabel('Mean NDVI')
ax.set_title('NDVI Recovery Trend: Rehabilitated vs Reference Vegetation\nMount Arthur Coal Mine, NSW')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

os.makedirs('/content/drive/MyDrive/GIS Portfolio/Project1_MineRehab/outputs', exist_ok=True)
plt.savefig('/content/drive/MyDrive/GIS Portfolio/Project1_MineRehab/outputs/ndvi_trend.png', dpi=300)
plt.show()



In [ ]:
# Cell 14 — Compute and display NDVI difference map between earliest and latest composite

early_ndvi = annual_composites[0].select('NDVI')
late_ndvi = annual_composites[-1].select('NDVI')
ndvi_change = late_ndvi.subtract(early_ndvi).rename('NDVI_change')

change_vis = {
    'min': -0.3, 'max': 0.3,
    'palette': ['red', 'white', 'green']
}

Map = geemap.Map()
Map.centerObject(aoi_fc, 14)
Map.addLayer(ndvi_change, change_vis, 'NDVI Change (latest - earliest)')
Map.addLayer(aoi_fc, {}, 'AOI zones')
Map


In [ ]:
# Cell 15 — Calculate percentage of each zone exceeding NDVI 0.4 vegetation threshold per year

def percent_vegetated(image, geometry, threshold=0.4):
    veg_mask = image.select('NDVI').gt(threshold)
    stats = veg_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=10,
        maxPixels=1e9
    ).getInfo()
    return stats.get('NDVI', 0) * 100

print('--- Rehab Mature ---')
for composite, year in zip(annual_composites, years):
    pct = percent_vegetated(composite, rehab_mature_geom)
    print(f'{year}: {pct:.1f}%')

print('--- Rehab Active ---')
for composite, year in zip(annual_composites, years):
    pct = percent_vegetated(composite, rehab_active_geom)
    print(f'{year}: {pct:.1f}%')

print('--- Reference ---')
for composite, year in zip(annual_composites, years):
    pct = percent_vegetated(composite, reference_geom)
    print(f'{year}: {pct:.1f}%')